# Preprocessing with BigQuery / BigFrames 

Just rewrote `03_preprocessing.ipynb`. Every step and import should be identical to the original, only the execution layer changes:
- `pd.read_csv(...)` to `bpd.read_gbq(...)` (data stays in BigQuery)
- `df[col].median()` computed inside BigQuery in one pass (not N queries)
- `pd.get_dummies(...)` to `bpd.get_dummies(...)` (pivot runs in BigQuery)
- `df.to_csv(...)` to `df.to_gbq(...)` (result written back to BigQuery)


## 0 - GCP configuration

Sets up BigQuery project and table references.

In [ ]:
# GCP / BigQuery settings
PROJECT_ID = 'mimic-iii-pipeline' 
DATASET    = 'mimic3_final_processed'              # BQ dataset that holds the tables
LOCATION   = 'europe-southwest1'                   # BQ processing location

# Mirrors the original CSV paths:
TABLE_FINAL_SELECTED  = f'{PROJECT_ID}.{DATASET}.final_selected'
TABLE_DIAGNOSIS_MAP   = f'{PROJECT_ID}.{DATASET}.diagnosis_map'
TABLE_FINAL_PROCESSED = f'{PROJECT_ID}.{DATASET}.final_processed'

## 1 - Imports 

In [ ]:
import bigframes.pandas as bpd
bpd.options.compute.ordering_mode = "partial"
import pandas as pd                              # kept for the small diagnosis_map lookup
from src.feature_groups_map import feature_groups    
from IPython.display import display

# Initialise BigFrames session
bpd.options.bigquery.project  = PROJECT_ID
bpd.options.bigquery.location = LOCATION

## 2 - Load data  *(mirrors original cell 2)*

```python
# original
df = pd.read_csv("data/datasets/final_selected.csv")
diagnosis_map_df = pd.read_csv("data/datasets/diagnosis_map.csv")
```

In [ ]:
df = bpd.read_gbq(TABLE_FINAL_SELECTED)

# diagnosis_map is small, pulling to pandas is fine
diagnosis_map_df = bpd.read_gbq(TABLE_DIAGNOSIS_MAP).to_pandas()
diagnosis_map_df = bpd.read_gbq(TABLE_DIAGNOSIS_MAP).to_pandas()
diagnosis_map_df.columns = ['diagnosis', 'category']

## 3 - Filter ICU deaths  *(mirrors original cell 4)*

In [ ]:
# Perform filter and drop
df = df[df["EXPIRE_FLAG"] == 0].drop(columns=["EXPIRE_FLAG"])

# Force BigQuery to save this state to a physical, temporary table
print("Saving filtered data to a temporary BigQuery table...")
df.to_gbq("mimic-iii-pipeline.mimic3_final_processed.temp_filtered_cohort", if_exists="replace")

# Reload it as a brand new, clean dataframe with no hidden computation history
print("Reloading clean data...")
df_clean = bpd.read_gbq("mimic-iii-pipeline.mimic3_final_processed.temp_filtered_cohort")

# Now run your missing info function on the clean dataframe

# Cache so BigQuery does not re-scan the raw table on every downstream operation
df = df.cache()

## 4 - Missing value analysis 

The aggregation runs inside BigQuery and only the small summary table is pulled locally.

In [ ]:
def get_missing_info(df):
    # isnull().sum() already computes per-column counts in one aggregation.
    null_counts = df.isnull().sum().to_pandas()
    total = null_counts.max()  # any non-null col gives the row count

    missing_info = pd.DataFrame({
        'null_count':      null_counts,
        'null_percentage': (null_counts / total * 100).round(4),
    }).sort_values('null_percentage', ascending=False)

    display(
        missing_info.style.set_table_attributes(
            'style="display:inline-block; max-height:500px; overflow:auto;"'
        )
    )
    return missing_info

In [ ]:
missing_info = get_missing_info(df)

Since the target variable of this project is Length of Stay (LOS), all rows with missing LOS values were removed from the dataset, as these samples cannot be used for supervised learning. Additionally, rows with missing diagnosis information were also excluded. As only a small number of records (approximately 17) lacked diagnosis data, removing them had minimal impact on the overall dataset while helping maintain data consistency during preprocessing.

## 5- Drop rows with missing LOS / DIAGNOSIS 


In [ ]:
df = df[df['LOS'].notnull()]
df = df[df['DIAGNOSIS'].notnull()]

To handle missing values, different imputation strategies were applied according to the semantic meaning of each feature group. Variables related to medications, interventions, procedures, outputs, and device usage were imputed with zero, since missing values in these cases often indicate that the event or intervention did not occur during the patient stay. On the other hand, continuous physiological and laboratory measurements were imputed using the median value of each feature. Median imputation was chosen because it is more robust to outliers and skewed distributions, which are common in clinical datasets such as MIMIC-III.

## 6 - Zero imputation 

In [ ]:
# Single plan node
zero_cols = {col: 0 for col in feature_groups['zero_impute'] if col in df.columns}
df = df.fillna(zero_cols)

## 7- Median imputation  

BigFrames computes each median inside BigQuery with `PERCENTILE_CONT`. To avoid N separate round-trips, all medians are fetched in a single aggregation pass first, then applied.

In [ ]:
# Cache after zero-imputation so BigQuery materialises the result once.
# All subsequent reads (including median agg) scan the cached table, not the raw one.
df = df.cache()

median_cols = [col for col in feature_groups['median_impute'] if col in df.columns]

# .median() on a DataFrame issues a single PERCENTILE_CONT over all columns.
# .to_pandas() does not pull the full table.
medians = df[median_cols].median().to_pandas()

df = df.fillna(medians.to_dict())

## 8 - Post-imputation missing check 

In [ ]:
missing_info = get_missing_info(df)

## 9 - Inspect categorical columns 

In [ ]:
categorical_columns = df.select_dtypes(include=['object', 'category', 'bool']).columns
print(categorical_columns)

After handling missing values, additional preprocessing steps were applied to prepare the categorical and temporal information for modeling. Since raw date variables are not directly suitable for machine learning algorithms, the patient's date of birth was transformed into an age feature by calculating the difference between admission time and birth date. This approach provides a more clinically meaningful representation of patient demographics. After generating the age variable, the original DOB and ADMITTIME columns were removed from the dataset to avoid redundancy and reduce unnecessary temporal information during training.

## 10 - Engineer age feature  

In [ ]:
# assign() is non-destructive and works directly on the cached frame.
df = df.assign(
    age=((bpd.to_datetime(df['ADMITTIME']) -
          bpd.to_datetime(df['DOB'])).dt.days / 365.25).astype(int)
).drop(columns=['DOB', 'ADMITTIME'])

The DIAGNOSIS column originally contained 15,248 unique categories, making it impractical to directly apply one-hot encoding or use the raw values for model training due to the extremely high dimensionality and sparsity that would be introduced into the dataset. To address this issue, a custom script named `src.create_diagnosis_map` was developed to automatically group diagnoses into a smaller set of clinically meaningful categories using a Large Language Model (LLM).

After generating the diagnosis-category mapping, the original diagnosis values were replaced by their corresponding grouped category using a dictionary-based mapping approach. This significantly reduced the cardinality of the feature while preserving clinically relevant information for the predictive modeling task.

## 11 - Count raw DIAGNOSIS categories 

In [ ]:
len(df['DIAGNOSIS'].value_counts())

## 12 - Map DIAGNOSIS to grouped categories 


In [ ]:
diagnosis_dict = dict(
    zip(diagnosis_map_df['diagnosis'], diagnosis_map_df['category'])
)
df['DIAGNOSIS'] = df['DIAGNOSIS'].map(diagnosis_dict)

## 13 - Count grouped DIAGNOSIS categories 

In [ ]:
len(df['DIAGNOSIS'].value_counts())

In [ ]:
df['DIAGNOSIS'].value_counts()

After that, one-hot encoding was applied to transform categorical variables into a numerical representation suitable for machine learning models. This technique converts each categorical value into a binary feature, allowing the algorithms to interpret categorical information without introducing artificial ordinal relationships between categories.

## 14 - One-hot encoding  

In [ ]:
df = bpd.get_dummies(
    df,
    columns=['DIAGNOSIS', 'ADMISSION_TYPE'],
    drop_first=True
)

# 2. Find the new dummy columns and cast them to integers (0/1)
dummy_cols = [col for col in df.columns if col.startswith('DIAGNOSIS_') or col.startswith('ADMISSION_TYPE_')]

for col in dummy_cols:
    df[col] = df[col].astype(int)

## 15 - Save processed dataset 

Writing back to BigQuery keeps the data on the cloud with zero cost for downstream training (Vertex AI, BigQuery ML, etc.).

In [ ]:
import os

df.to_gbq(
    TABLE_FINAL_PROCESSED,
    if_exists='replace',
)

print(f'Written to: {TABLE_FINAL_PROCESSED}')
print('Final shape:', df.shape)

os.makedirs("data/datasets", exist_ok=True)

# Save as a Parquet file
# Fast to read/write, smaller file size, and preserves data types
df.to_parquet("data/datasets/final_encoded.parquet", index=False)

# Save as a CSV file
# Easy to open in Excel or view manually, but takes more space
df.to_csv("data/datasets/final_encoded.csv", index=False)

print("Data saved successfully!")